In [1]:
import math

# -------- PRINT BOARD ----------
def print_board(board, n):
    for r in range(n):
        row = []
        for c in range(n):
            row.append(board[r][c] if board[r][c] != " " else "_")
        print(" ".join(row))
    print()

In [1]:
# -------- EMPTY CELLS ----------
def empty_cells(board, n):
    cells = []
    for r in range(n):
        for c in range(n):
            if board[r][c] == " ":
                cells.append((r, c))
    return cells

In [3]:
# -------- CHECK WINNER ----------
def winner(board, n, k):
    # Directions to check: Right, Down, Down-Right, Down-Left
    directions = [(0, 1), (1, 0), (1, 1), (1, -1)]
    
    for r in range(n):
        for c in range(n):
            if board[r][c] != " ":
                player = board[r][c]
                for dr, dc in directions:
                    if 0 <= r + dr*(k-1) < n and 0 <= c + dc*(k-1) < n:
                        match = True
                        for i in range(1, k):
                            if board[r + dr*i][c + dc*i] != player:
                                match = False
                                break
                        if match:
                            return player
    return None

In [4]:
# -------- DRAW CHECK ----------
def is_draw(board, n, k):
    return winner(board, n, k) is None and len(empty_cells(board, n)) == 0


In [5]:
def evaluate_board(board, n, k):
    score = 0
    lines = []

    # Collect all rows
    for r in range(n):
        lines.append(board[r])

    # Collect all columns
    for c in range(n):
        lines.append([board[r][c] for r in range(n)])

    # Collect diagonals
    for d in range(-(n - k), n - k + 1):
        lines.append([board[r][r - d] for r in range(n) if 0 <= r - d < n])
        lines.append([board[r][n - 1 - r + d] for r in range(n) if 0 <= n - 1 - r + d < n])

    # Score each line
    for line in lines:
        if len(line) >= k:
            for i in range(len(line) - k + 1):
                window = line[i:i + k]
                if window.count("O") > 0 and window.count("X") == 0:
                    score += (window.count("O") ** 2)
                elif window.count("X") > 0 and window.count("O") == 0:
                    score -= (window.count("X") ** 2)

    return score


In [6]:
# -------- MINIMAX WITH ALPHA-BETA PRUNING ----------
def minimax(board, depth, alpha, beta, is_maximizing, n, k):
    w = winner(board, n, k)

    if w == "O":   
        return 10 + depth  
    if w == "X":   
        return -10 - depth 
    if is_draw(board, n, k):
        return 0
        
    if depth == 0:         
        return evaluate_board(board,n,k)

    if is_maximizing:
        best = -math.inf
        for (r, c) in empty_cells(board, n):
            board[r][c] = "O"
            score = minimax(board, depth - 1, alpha, beta, False, n, k)
            board[r][c] = " "
            
            best = max(best, score)
            alpha = max(alpha, best)
            
            if beta <= alpha:
                break 
        return best
        
    else:
        best = math.inf
        for (r, c) in empty_cells(board, n):
            board[r][c] = "X"
            score = minimax(board, depth - 1, alpha, beta, True, n, k)
            board[r][c] = " "
            
            best = min(best, score)
            beta = min(beta, best)
            
            if beta <= alpha:
                break 
        return best


In [7]:
# -------- BEST MOVE ----------
def best_move(board, n, k):
    best_score = -math.inf
    move = ()
    
    max_depth = 5 if n > 3 else 9 

    for (r, c) in empty_cells(board, n):
        board[r][c] = "O"
        score = minimax(board, max_depth - 1, -math.inf, math.inf, False, n, k)
        board[r][c] = " "

        if score > best_score:
            best_score = score
            move = (r, c)

    return move

In [2]:
# -------- MAIN GAME ----------
if __name__ == "__main__":
    print("--- Dynamic Tic Tac Toe (Alpha-Beta) ---")
    
    try:
        N = int(input("Enter board size N (e.g., 3 for 3x3, 4 for 4x4): "))
        K = int(input(f"Enter win condition K (e.g., 3 for 3-in-a-row): "))
    except ValueError:
        print("Invalid input. Defaulting to 3x3 with 3-in-a-row.")
        N, K = 3, 3

    if K > N:
        print(f"Warning: Win condition ({K}) is larger than the board ({N}). Reducing K to {N}.")
        K = N

    board = [[" " for _ in range(N)] for _ in range(N)]

    print(f"\nGame Start: {N}x{N} Board, {K} in a row to win.")
    print_board(board, N)

    while True:
        try:
            r = int(input(f"Enter row (0 to {N-1}): "))
            c = int(input(f"Enter col (0 to {N-1}): "))
        except ValueError:
            print("Please enter valid numbers.")
            continue

        if r < 0 or r >= N or c < 0 or c >= N or (r, c) not in empty_cells(board, N):
            print("Invalid move. Try again.")
            continue

        board[r][c] = "X"
        print("\nAfter your move:")
        print_board(board, N)

        if winner(board, N, K) == "X":
            print("You win!")
            break

        if is_draw(board, N, K):
            print("Draw!")
            break

        print("Computer is thinking...")
        mr, mc = best_move(board, N, K)
        if mr is not None and mc is not None:
            board[mr][mc] = "O"

        print("Computer move:")
        print_board(board, N)

        if winner(board, N, K) == "O":
            print("Computer wins!")
            break

        if is_draw(board, N, K):
            print("Draw!")
            break

--- Dynamic Tic Tac Toe (Alpha-Beta) ---


Enter board size N (e.g., 3 for 3x3, 4 for 4x4):  4
Enter win condition K (e.g., 3 for 3-in-a-row):  3



Game Start: 4x4 Board, 3 in a row to win.


NameError: name 'print_board' is not defined